# Scoring logos with TRIBE v2

[TRIBE v2](https://huggingface.co/facebook/tribev2) predicts fMRI brain responses to naturalistic stimuli (video/audio/text). Here we treat each logo as a silent visual stimulus, get TRIBE v2's predicted brain-response vector for it, then use PCA to place all logos in a 2D space to see how they relate to each other.

In [ ]:
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(".env")
import os

if os.environ.get("HUGGING_FACE_TOKEN"):
    login(token=os.environ["HUGGING_FACE_TOKEN"])

LOGOS_DIR = Path("logos")
CACHE_FOLDER = Path("./cache")
logo_paths = sorted(LOGOS_DIR.glob("*.png")) + sorted(LOGOS_DIR.glob("*.jpeg"))
logo_paths

## Load TRIBE v2

Downloads the checkpoint from Hugging Face on first run (~1GB).

In [ ]:
from tribev2.demo_utils import TribeModel

model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=CACHE_FOLDER)

## Turn each logo into a short silent clip

TRIBE v2 only accepts video/audio/text files, not static images, so each logo is rendered as a short looping clip (no audio track).

In [ ]:
from moviepy import ImageClip

CLIP_DURATION = 6.0
video_paths = {}
for logo_path in logo_paths:
    video_path = CACHE_FOLDER / f"{logo_path.stem}.mp4"
    clip = ImageClip(str(logo_path), duration=CLIP_DURATION).resized(height=256)
    clip.write_videofile(str(video_path), codec="libx264", audio=False, fps=8, logger=None)
    video_paths[logo_path.stem] = video_path
video_paths

## Score each logo

For each clip we build the events dataframe, run `model.predict`, and average the predicted brain response over time to get one embedding vector per logo.

In [ ]:
embeddings = {}
for name, video_path in video_paths.items():
    print(f"Scoring {name}...")
    events = model.get_events_dataframe(video_path=video_path)
    preds, segments = model.predict(events=events, verbose=False)
    embeddings[name] = preds.mean(axis=0)

names = list(embeddings.keys())
X = np.stack([embeddings[name] for name in names])
X.shape

## PCA to 2D and plot

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from PIL import Image
from sklearn.decomposition import PCA

coords = PCA(n_components=2).fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(coords[:, 0], coords[:, 1], s=0)
for (x, y), logo_path in zip(coords, logo_paths):
    img = Image.open(logo_path).convert("RGBA")
    ab = AnnotationBbox(OffsetImage(img, zoom=0.15), (x, y), frameon=False)
    ax.add_artist(ab)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Logos placed by TRIBE v2 predicted brain response (PCA)")
plt.show()